In [1]:
# new_selector.xpath("//a[@class='inline-block group']/@href").getall() //get all the links on a page
# new_selector.xpath("//h1[@class='font-bold type-preset-2']/text()").get() //name of the faculty 
# new_selector.xpath("//dl[@class='mt-8']//text()").getall() // department of the faculty
# new_selector.xpath("//div[@class='border-t border-slate-100 text-blue-400']//a/@href").getall() //link to personal page
# new_selector.xpath("//div[@class='gutenberg-editor']/p/text()").getall() // info of the faculty
# new_selector.xpath("//div[@class='pagination text-center']/a/@href").get() // next page

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from scrapy import Selector
driver_path = r'.\chromedriver.exe'
brave_path = r'C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe'

service = Service(driver_path)
option = webdriver.ChromeOptions()
option.binary_location = brave_path
browser = webdriver.Chrome(service=service, options=option)
browser.get("https://datascience.columbia.edu/people-type/faculty/")
new_selector = Selector(text=browser.page_source)

In [ ]:
with open('DSI Faculty.txt', "w", encoding="utf-8") as w:
    while next_page:=new_selector.xpath("//div[@class='pagination text-center']/a[@class='next page-numbers']/@href").get():
        for faculty in new_selector.xpath("//a[@class='inline-block group']/@href").getall():
            browser.get(faculty)
            info_selector = Selector(text=browser.page_source)
            name = info_selector.xpath("//h1[@class='font-bold type-preset-2']/text()").get() # name of the faculty 
            dept = info_selector.xpath("//dl[@class='mt-8']//text()").getall() # department of the faculty
            links = info_selector.xpath("//div[@class='border-t border-slate-100 text-blue-400']//a/@href").getall() # link to personal page
            info = info_selector.xpath("//div[@class='gutenberg-editor']/p/text()").getall() # info of the faculty
            w.write(f'{name}\n')
            for line in dept:
                w.write(f"{line.strip()}\n")
            for link in links:
                w.write(f'\n{link}')
            w.write('\n\n')
            for i in info:
                w.write(f"{i}")
            w.write('\n\n\n==============================================================================================================================================\n\n\n')
        browser.get(next_page)
        print(next_page)
        new_selector = Selector(text=browser.page_source)

### Information Retrival

In [8]:
%%content
!pip install chromadb --no-deps
!pip install sentence-transformers==5.1.2
!pip install transformers==4.57.1
!pip install langchain langchain-community langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 97.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 29.2 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.2.0
    Uninstalling sentence-transformers-5.2.0:
      Successfully uninstalled sentence-transformers-5.2.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 110.6 MB/s eta 0:00:0000:010:01
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB

In [ ]:
import os
from pathlib import Path
import transformers
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import AutoTokenizer
import langchain
import chromadb
from chromadb.config import Settings
from rich.traceback import install; install()
import pandas as pd 
import os
from dotenv import load_dotenv
from huggingface_hub import login
import torch

load_dotenv() # load the variables from .env file
hf_token = os.environ.get("HF_TOKEN") 
login(token=hf_token) # login to hugging face; though not necessary for open models

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
)

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    device_map="auto", # this will put the device on cuda by default
    dtype="auto", # keeps weights in 16 bit
)

sent_model_name = "sentence-transformers/all-mpnet-base-v2"
embedder = SentenceTransformer(sent_model_name, device=device)
mpnet_tokenizer = AutoTokenizer.from_pretrained(sent_model_name)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def chunk_text_token_overlap(text, chunk_size=300, overlap=50):
    # encode sentence to tokens
    # add_special_tokens is set to false otherwise overlapping tokens will be 
    # different from the sentence because a special BOS/EOS token is added
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []

    start = 0
    end = chunk_size

    while start < len(tokens):
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)

        start = end - overlap
        end = start + chunk_size

    return chunks

In [6]:
faculty_df = pd.read_json("/content/dsi_faculty.json", encoding='latin8')

In [7]:
faculty_df.head()

,name,dept,links,info
0,Ryan Abernathey,Faculty of Arts and Sciences\nAssociate Profes...,"[mailto:rpa@ldeo.columbia.edu, https://raberna...",Ryan P. Abernathey is an Associate Professor o...
1,Paris Adkins-Jackson,Mailman School of Public Health\nAssistant Pro...,"[mailto:pa2629@cumc.columbia.edu, https://www....","Paris AJ Adkins-Jackson, PhD MPH is a multid..."
2,Anish Agarwal,Columbia Engineering\nAssistant Professor of I...,"[mailto:aa5194@columbia.edu, https://sites.goo...",Anishs research interests are in causal infer...
3,Shipra Agrawal,Columbia Engineering\nCyrus Derman Associate P...,"[mailto:sa3305@columbia.edu, http://www.columb...",Professor Shipra Agrawal is the Cyrus Derman A...
4,Sunil Agrawal,Columbia Engineering\nProfessor of Mechanical ...,"[mailto:sunil.agrawal@columbia.edu, http://roa...",Dr. Agrawal obtained a PhD degree in Mechanica...


In [38]:
client = chromadb.PersistentClient(path="/content/drive/MyDrive/db/")
collection = client.get_or_create_collection(
    name="faculty_info",
    metadata={"hnsw:space": "cosine"}
)

In [2]:
def store_info():
    for idx, row in faculty_df.iterrows():
        # create chunks for the faculty information
        chunks = chunk_text_token_overlap(row['info'])

        # generate embeddings
        embeddings = embedder.encode(
            chunks,
            batch_size=32,
            convert_to_numpy=True,
            # all-mpnet-base-v2 normalizes by default;
            # we need normalized vector for comparision
            normalize_embeddings=False, # just setting flag explicitely 
        ) # embeddings shape: (n_chunks, 768)

        # add each chunk to the vector db
        for i, chunk in enumerate(chunks):
            collection.add(
                documents=[chunk],
                embeddings=[embeddings[i]],
                metadatas=[{
                    "name": row["name"],
                    "department": row["dept"],
                    "chunk_index": i,
                }],
                ids=[f"{row["name"]}_info_{i}"], # add some indentifier
            )

def generate_answer_mistral(prompt, max_tokens=256):
    # tokenize the prompt before feeding to the LLM
    # tokenizer returns a hg dictionary wrapper that has a .to method
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    output = model.generate(
        **inputs, # BatchEncoding object
        max_new_tokens=max_tokens, # stop after generating this many *new* toeksn
        do_samples=False,  # greedy decoding (deterministic)
    ) 

    return tokenizer.decode(output[0], skip_special_tokens=True) # skip </s> <s>

def build_rag_prompt(query, retrieved_chunks):
    prompt = "You are a factual assistant. Use ONLY the evidence below.\n"
    prompt += "If the answer is not in the evidence, say 'NOT FOUND in the given evidence'\n\n"

    prompt += "EVIDENCE:\n"
    for i, chunk in enumerate(retrieved_chunks):
        text = chunk["text"]
        metadata = chunk["metadata"]
        prompt += f"[{i}] {text}\n"
        prompt += f"Metadata: {metadata}\n\n"
    
    prompt += f"QUESTION: {query}\n\n"
    prompt += "ANSWER (cite evidence like [0], [1], etc). Please provide only one concise answer.\n"

    return prompt

def retrive_and_answer(query, n_results=4):
    # create query vector from query text
    query_emb = embedder.encode(query)

    # retrive the results
    results = collection.query(
        query_embedding=query_emb,
        n_results=n_results
    )

    # stitch together fetched data to provide it for generation
    retrieved = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        retrieved.append({"text": doc, "metadata": meta}) 
    
    prompt = build_rag_prompt(query, retrieved)
    answer = generate_answer_mistral(prompt, max_tokens=256)
    return answer

In [ ]:
store_info()

In [3]:
query = "Which professor works on earth science?"
retrive_and_answer(query)

NameError: name 'embedder' is not defined